# 🌲 Notebook 01 — ÚLOHY: Průzkum a příprava dat

---

## Jak na to
1. Spusť nejprve setup buňku (⚙️ Setup níže)
2. Přečti zadání každé úlohy
3. Napiš vlastní kód do buňky označené `# TVŮJ KÓD ZDE`
4. Zkontroluj svůj výstup s očekávaným výsledkem
5. **Řešení** je v souboru `01_pruzkum_dat_RESENI.ipynb` — otevři ho **až po vlastním pokusu!**
6. Nápovědy jsou skryté v komentářích — odkomentuj je pokud potřebuješ

## Cíle
- Načíst 3 CSV soubory pomocí pandas
- Prozkoumat data (typy, chybějící hodnoty, statistiky)
- Spojit tabulky do jednoho datasetu
- Exportovat výsledek

---
## ⚙️ Setup

Tuto buňku spusť vždy jako první — nastaví prostředí.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings, os
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)
print('Setup hotový!')

---
## Úloha 1: Načtení EKC datasetu

**Zadání**: Načti soubor `../../dan_data/forest_ekc_model.csv` do DataFrame s názvem `forest_ekc`.
Poté vypiš:
- Počet zemí (řádků) a sloupců
- Názvy sloupců
- Prvních 5 řádků

**Očekávaný výstup**:
```
Počet zemí: 200+
Sloupce: ['Country', 'Code', 'Forest_1990', 'Forest_2025', 'Forest_change']
```

> 💡 **Nápověda**: Použij `pd.read_csv()`, `.shape`, `.columns.tolist()`, `.head()`

> 📁 **Proč `../../`?** Notebook leží v `Kuznets_analysis/A_ulohy/` — každé `../` posune o jednu složku výš. Dvě úrovně výš: `A_ulohy/ → Kuznets_analysis/ → digital_academy_project/`, kde se nachází složka `dan_data/`.

In [ ]:
# TVŮJ KÓD ZDE
# forest_ekc = pd.read_csv('../../dan_data/forest_ekc_model.csv')

---
## Úloha 2: Průzkum dat

**Zadání**: Prozkoumej dataset `forest_ekc`:
1. Zobraz datové typy a počet chybějících hodnot pomocí `.info()`
2. Zobraz popisnou statistiku pomocí `.describe()`
3. Zjisti kolik zemí **zalesňuje** (Forest_change > 0) a kolik **odlesňuje** (Forest_change < 0)

> 💡 **Nápověda k bodu 3**: `(df['sloupec'] > 0).sum()` spočítá počet řádků kde je podmínka True

**Očekávaný výstup (bod 3):**
```
Zalesňující: 90
Odlesňující: 91
Bez změny:   30
```

In [ ]:
# TVŮJ KÓD ZDE — .info() a .describe()
# forest_ekc.info()

In [ ]:
# TVŮJ KÓD ZDE — počet zalesňujících vs. odlesňujících zemí
# zalesnovani = (forest_ekc['Forest_change'] > 0).sum()

---
## Úloha 3a: Načtení a příprava HDP dat

**Zadání** (kroky 1–3):
1. Načti `../../dan_data/MAIN_Forest_GDP_joined.csv` do DataFrame `forest_gdp`
2. Přejmenuj sloupce:
   - `'Share of land covered by forest'` → `'forest_pct'`
   - `'GDP per capita (current US$)'` → `'gdp_per_capita'`
3. Filtruj na roky >= 1990 → ulož jako `forest_gdp_filtered`

> 📋 **Struktura dat**: `MAIN_Forest_GDP_joined.csv` je **panelový dataset** — každá země má **více řádků**, jeden pro každý rok (1990–2024). Celkem ~7 000 řádků pro ~200 zemí × ~35 let. Abychom dostali jedno číslo HDP na zemi, budeme v Úloze 3b seskupovat (`groupby`) podle `Entity` a `Code` a brát průměr přes roky.

> 💡 **Nápověda**:
> - `.rename(columns={'stary_nazev': 'novy_nazev'})` přejmenuje sloupce
> - `df[df['Year'] >= 1990]` filtruje řádky

In [ ]:
# TVŮJ KÓD ZDE — kroky 1, 2, 3
# forest_gdp = pd.read_csv('../../dan_data/MAIN_Forest_GDP_joined.csv')

---
## Úloha 3b: Průměrné HDP na zemi

**Zadání** (kroky 4–5):
4. Vypočítej průměrné HDP pro každou zemi pomocí `.groupby()`:
   → ulož jako `mean_gdp` s přejmenováním: Entity→Country, gdp_per_capita→mean_gdp
5. Vypiš kolik zemí výsledný DataFrame obsahuje

> 💡 **Nápověda**:
> - `.groupby(['Entity', 'Code'])['gdp_per_capita'].mean().reset_index()`
> - `.rename(columns={'Entity': 'Country', 'gdp_per_capita': 'mean_gdp'})`

> 📝 **Proč průměr a ne HDP za konkrétní rok?** Změna lesa probíhala **postupně** v průběhu 35 let — proto průměrné HDP lépe zachytí ekonomickou úroveň v celém sledovaném období než snapshot jediného roku. Alternativy (HDP roku 2000, medián) dávají podobné výsledky. ⚠️ Omezení: zemím s rychlým růstem (Vietnam, Čína) přiřadíme nižší HDP, než mají dnes.

**Očekávaný výstup:**
```
Průměrné HDP pro 214 zemí
```

In [ ]:
# TVŮJ KÓD ZDE — krok 4 a 5: průměrné HDP
# mean_gdp = (

---
## Úloha 3c: Vývoj lesního pokryvu v čase

**Zadání**: Z panelového datasetu `forest_gdp_filtered` zjisti, jak se měnil průměrný % lesního pokryvu v čase (1990–2024):
1. Pomocí `.groupby('Year')['forest_pct'].agg(mean='mean', median='median', count='count').reset_index().round(3)` spočítej průměr a medián za každý rok
2. Vypiš hodnoty sloupce `mean` pro roky 1990, 2000, 2010, 2020, 2024
3. Graf je připraven v buňce níže — spusť ho a ověř, že trend odpovídá hodnotám z bodu 2

> ⚠️ **Metodická poznámka**: Toto je **nevážený průměr** — každá země (Monaco i Rusko) má stejnou váhu bez ohledu na rozlohu. Nezobrazuje skutečné globální procento plochy lesa, ale typickou hodnotu pro průměrnou zemi.

> 💡 **Nápověda**: Seskupuješ podle 1 sloupce (Year), ne 2 (Entity + Code).

> 📌 Pokračuj s **Úlohou 3d** — tam spočítáme skutečné globální % vážené rozlohou zemí.

In [ ]:
# TVŮJ KÓD ZDE
# yearly_forest = (

In [ ]:
# Grafická kontrola — čárový graf (kód připraven předem)
# Spusť tuto buňku až po dokončení kroků 1–2 výše (musí existovat: yearly_forest)

if 'yearly_forest' not in dir():
    print('⚠️  Nejdřív dokonči předchozí buňku — yearly_forest není definováno!')
elif 'mean' not in yearly_forest.columns and 'forest_pct' not in yearly_forest.columns:
    print('⚠️  yearly_forest musí mít sloupec "mean" (z .agg()) nebo "forest_pct" (z .mean()).')
    print('    Použij: .agg(mean="mean", median="median", count="count").reset_index()')
else:
    y_col = 'mean' if 'mean' in yearly_forest.columns else 'forest_pct'
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(yearly_forest['Year'], yearly_forest[y_col], 'b-', linewidth=2, label='Průměr')
    if 'median' in yearly_forest.columns:
        ax.plot(yearly_forest['Year'], yearly_forest['median'], 'g--', linewidth=1.5, label='Medián')
        ax.fill_between(yearly_forest['Year'], yearly_forest[y_col], yearly_forest['median'], alpha=0.1)
        ax.legend()
    ax.set_xlabel('Rok')
    ax.set_ylabel('% lesního pokryvu')
    ax.set_title('Světový průměr % lesního pokryvu 1990–2024')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print('\n➡️  Pokud průměr klesá, globálně přichází o les. Pokud roste, globálně zalesňujeme.')

---
## Úloha 3d: Skutečné globální % lesa — vážený průměr

**Zadání**: Soubor `../../dan_data/Forest area compared to total area.csv` obsahuje absolutní plochu lesa a celkovou rozlohu zemí (v 1 000 ha) pro roky 1990, 2000, 2010, 2015, 2020, 2025. Spočítej skutečné globální procento lesního pokryvu:

1. Načti soubor s dvouřádkovým záhlavím: `pd.read_csv('...', header=[0, 1], index_col=0)`
2. Pro každý rok spočítej vážený průměr:
   `weighted_pct = forest_sum / land_sum * 100`
3. Porovnej výsledek 1990 a 2025 s nevážným průměrem z `forest_ekc` (Forest_1990, Forest_2025)
4. Vypiš výsledky pro všechny roky

> ⚠️ **Dvouřádkové záhlaví**: Řádek 0 = typ dat (`Forest (1 000 ha)` nebo `Total land area (1 000 ha)`), řádek 1 = rok (`'1990'`, `'2000'`, ...). Jsou to **řetězce**, ne čísla.

> 💡 **Nápověda**: Po načtení: `fa['Forest (1 000 ha)']['1990'].dropna().sum()` → celková plocha lesa v roce 1990.

**Očekávaný výstup:**
```
Skutečný vážený průměr lesního pokryvu:
1990: 33.33%
2000: 32.51%
2010: 32.24%
2015: 32.09%
2020: 31.96%
2025: 31.79%

Srovnání s nevážným průměrem per-country:
  1990: nevážený 34.18% vs. vážený 33.33% → -0.86 pp
  2025: nevážený 32.97% vs. vážený 31.79% → -1.18 pp
```

> 🔍 **Klíčový výsledek**: Vážený průměr je nižší než nevážený (~1 pp), protože velké, hustě zalesněné země (Rusko, Kanada, Brazílie) mají ve váženém průměru větší vliv. Obě řady ale ukazují shodný klesající trend — svět odlesňuje tempem ~1.5 pp za 35 let.

In [ ]:
## TVŮJ KÓD ZDE
#fa = pd.read_csv('../../dan_data/Forest area compared to total area.csv',

---
## Úloha 4: Načtení klasifikace Světové banky

**Zadání**:
1. Načti `../../dan_data/2025_World_Bank_classification_by_Income.csv` do `wb_raw`
   ⚠️ Použij `skiprows=2` — soubor má **2 řádky metadat** před záhlavím!
2. Vypiš sloupce a prvních 5 řádků
3. Vyber sloupce `Code`, `Region`, `Income group` → ulož jako `wb_class`
4. Přejmenuj je na malá písmena: `code`, `region`, `income_group`
5. Odstraň řádky kde `code` nebo `income_group` je NaN
6. Filtruj pouze validní ISO3 kódy: `wb_class = wb_class[wb_class['code'].str.match(r'^[A-Z]{3}$', na=False)]`
7. Vypiš počet zemí a hodnoty v `income_group` (`.value_counts()`)

> 💡 **Proč `skiprows=2`?** Otevři CSV v Excelu — první dva řádky jsou metadata
> Světové banky, teprve třetí řádek je záhlaví sloupců. S `skiprows=1` dostaneš `KeyError: 'Code'`.

> 💡 **Proč `str.match(r'^[A-Z]{3}$')`?** Dataset Světové banky obsahuje kromě zemí
> i řádky pro regiony a agregáty (`WLD`, `EUU`, `LCN`...) — všechna písmena velká, právě
> 3 znaky. Jenže tam jsou i řádky jako `Channel Islands` nebo prázdné buňky, které
> by merge zkomplikovaly. Regex `^[A-Z]{3}$` znamená: začátek (`^`), přesně 3 velká
> písmena (`[A-Z]{3}`), konec (`$`). `na=False` ignoruje NaN bez chyby.

**Očekávaný výstup (bod 7):**
```
Počet zemí: 218
income_group
High income            86
Upper middle income    55
Lower middle income    51
Low income             26
```

In [ ]:
# TVŮJ KÓD ZDE
# wb_raw = pd.read_csv('../../dan_data/2025_World_Bank_classification_by_Income.csv', skiprows=2)

---
## Úloha 5a: Spojení tabulek (Merge)

**Zadání** (kroky 1–4):
1. Začni s `forest_ekc`
2. Přidej `mean_gdp[['Code', 'mean_gdp']]` pomocí `pd.merge()` s `on='Code'`, `how='left'`
3. Přidej `wb_class` — pozor: klíče mají **různou velikost písmen**:
   → Použij `left_on='Code', right_on='code'`
4. Hned po merge odstraň přebytečný klíč `code` z wb_class:
   `ekc_master = ekc_master.drop(columns=['code'])`

> 💡 **Proč jiné klíče pro druhý merge?**
>
> | Tabulka | Sloupec pro join |
> |---------|-----------------|
> | `forest_ekc` | `Code` (velká C) |
> | `mean_gdp` | `Code` (velká C) → stejný název, stačí `on='Code'` |
> | `wb_class` | `code` (malá c) → různé názvy, nutno `left_on='Code', right_on='code'` |

> 💡 **Proč drop hned v kroku 4?** `pd.merge(..., left_on='Code', right_on='code')` zachová v výsledku **oba** join klíče. Pokud `code` neodstraníš teď a v Úloze 5b přejmenuješ `Code` → `code`, budeš mít dva sloupce `code` — a pandas pak vyhodí záhadnou chybu o deset kroků dál.

> 💡 **Nápověda**: `pd.merge(ekc_master, wb_class, left_on='Code', right_on='code', how='left')`

In [ ]:
# TVŮJ KÓD ZDE — merge kroky 1-4
# ekc_master = pd.merge(

---
## Úloha 5b: Vyčistění a doplnění datasetu

**Zadání** (kroky 5–8):
5. Přejmenuj sloupce na malá písmena: Country→country, Code→code, Forest_1990→forest_1990, atd.
6. Odstraň řádky kde chybí `forest_change`, `mean_gdp` nebo `income_group` → ulož jako `ekc_complete`
7. Přidej sloupec `log_gdp = np.log(mean_gdp)`
8. Vypiš počet zemí v `ekc_complete`

> 💡 **Nápověda k bodu 5**: `df.rename(columns={'Country': 'country', 'Code': 'code', ...})`
>
> 💡 **Nápověda k bodu 6**: `.dropna(subset=['sloupec1', 'sloupec2', ...])'

> ✅ **Kontrolní bod**: Po `dropna` by měl `ekc_complete` obsahovat přibližně **199 zemí**. Pokud máš výrazně méně, zkontroluj sloupce v merge kroku výše — nejspíš chybí `income_group`.

**Očekávaný výstup (bod 8):**
```
Kompletní záznamy: 199 zemí
```

In [ ]:
# TVŮJ KÓD ZDE — přejmenování, dropna, log_gdp
# ekc_master = ekc_master.rename(columns={

---
## Úloha 6: Průměrná změna lesa podle příjmové skupiny

**Zadání**: Pomocí `.groupby()` spočítej pro každou příjmovou skupinu:
- průměr, medián, std. odchylku a počet zemí pro sloupec `forest_change`

**Bonus**: Zobraz výsledky jako sloupcový graf (barplot) s chybovými úsečkami.

> 💡 **Nápověda**: `.groupby('income_group')['forest_change'].agg(['mean', 'median', 'std', 'count'])`

**Očekávaný výstup (formát):**
```
                     Průměr  Medián   Std  Počet zemí
income_group
Low income           -5.547  -2.880  7.005          25
Lower middle income  -3.911  -1.140  8.631          47
Upper middle income  -0.608   0.000  7.406          52
High income           1.383   0.790  4.287          75
```

> 🔍 **Klíčový výsledek**: Jako jediná skupina má **High income** kladný průměr (+1.383 %) — bohatší státy průměrně zalesňují, chudé státy průměrně odlesňují.

In [ ]:
# TVŮJ KÓD ZDE
# income_order = ['Low income', 'Lower middle income', 'Upper middle income', 'High income']

---
## Bonus: Sanity Check — Ověření správnosti dat

**Zadání**: Ověř, že tvůj `ekc_complete` dataset je správně sestavený:
1. Vyber řádky pro tyto ISO kódy: `['CZE', 'BRA', 'ETH', 'DEU']`
2. Pro každou zemi vypiš: `country`, `income_group`, `forest_change`, `mean_gdp`
3. Zkontroluj, že hodnoty dávají smysl (Německo = High income, Etiopie = Low income)
4. Spočítej kolik zemí je v každé příjmové skupině (`.value_counts()`)

In [ ]:
# NENÍ TŘEBA PSÁT SAMOSTATNĚ - STAČÍ SPUSTIT
# NA KONCI ZKONTROLUJ, ZDA VÝSLEDKY DÁVAJÍ SMYSL
test_codes = ['CZE', 'BRA', 'ETH', 'DEU', 'IDN']
print('=== SANITY CHECK ===')
for code in test_codes:
    row = ekc_complete[ekc_complete['code'] == code]
    if len(row) == 0:
        print(f'CHYBA: {code} nenalezen! Zkontroluj merge.')
    else:
        r = row.iloc[0]
        print(f'{code} ({r["country"]:20s}) | income: {str(r.get("income_group","?")):<22} | '
              f'forest_change: {r["forest_change"]:+6.2f}% | HDP: ${r["mean_gdp"]:>9,.0f}')

print('\nRozložení příjmových skupin:')
print(ekc_complete['income_group'].value_counts())

---
## Úloha 7: Export

**Zadání**: 
1. Vytvoř složku `../output/` (použij `os.makedirs('../output', exist_ok=True)`)
2. Exportuj `ekc_complete` do `../output/ekc_analysis.csv` bez index sloupce
   - Použij `encoding='utf-8-sig'` (pro správné zobrazení češtiny)

**Očekávaný výstup:**
```
✅ Uloženo: ../output/ekc_analysis.csv (199 řádků)
```

> ✅ **Ověření**: V Průzkumníku souborů (nebo `ls ../output/`) musí soubor `ekc_analysis.csv` existovat. Pokud ho otevřeš v Excelu, sloupce by měly být: `country`, `code`, `forest_1990`, `forest_2025`, `forest_change`, `mean_gdp`, `region`, `income_group`, `log_gdp`.

In [ ]:
# TVŮJ KÓD ZDE
# os.makedirs('../output', exist_ok=True)

---
## ✅ Hotovo?

Porovnej svůj výsledek s řešením v `01_pruzkum_dat_RESENI.ipynb`.

Pokud tvůj výsledek:
- Má stejný počet zemí ✓
- Obsahuje správné sloupce ✓  
- Soubor `ekc_analysis.csv` byl vytvořen ✓

→ **Pokračuj s** `02_statistika_ekc_ULOHY.ipynb`
